### Import libraries

In [0]:
import hashlib
import json
import math
import re
import time
from datetime import datetime, timedelta, timezone
from email.utils import parsedate_to_datetime

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from delta.tables import DeltaTable
from pyspark.sql import functions as F

### Set variables

In [0]:
CATALOG = "dbw_dariusz"
SCHEMA = "bronze"

# Required: real company email or a website with contact information.
APPLICATION_CONTACT = "dkrol@hotmail.no"

LOCATIONS = [
    {
        "location_id": "stavanger",
        "latitude": 58.9700,
        "longitude": 5.7331,
    },
]

API_URL = "https://api.met.no/weatherapi/locationforecast/2.0/compact"
USER_AGENT = f"WeatherForecastIngestion/1.0 {APPLICATION_CONTACT}"

REQUEST_TIMEOUT = (10, 60)
REQUEST_INTERVAL_SECONDS = 1
CACHE_FALLBACK_MINUTES = 10

RAW_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`met_norway_weather_raw`"
FORECAST_VIEW = f"`{CATALOG}`.`{SCHEMA}`.`met_norway_weather_forecasts`"

RAW_SCHEMA = """
    record_id STRING,
    request_key STRING,
    location_id STRING,
    latitude DOUBLE,
    longitude DOUBLE,
    source_url STRING,
    source_updated_at TIMESTAMP,
    first_ingested_at TIMESTAMP,
    last_checked_at TIMESTAMP,
    expires_at TIMESTAMP,
    last_modified STRING,
    http_status INT,
    raw_payload STRING
"""

FORECAST_JSON_SCHEMA = """
    STRUCT<properties: STRUCT<
        meta: STRUCT<units: MAP<STRING, STRING>>,
        timeseries: ARRAY<STRUCT<
            time: STRING,
            data: MAP<STRING, STRUCT<
                details: MAP<STRING, DOUBLE>,
                summary: MAP<STRING, STRING>
            >>
        >>
    >>
"""

### Common functions

In [0]:
def utc_now():
    return datetime.now(timezone.utc)


def parse_http_date(value):
    if not value:
        return None

    result = parsedate_to_datetime(value)

    if result.tzinfo is None:
        result = result.replace(tzinfo=timezone.utc)

    return result.astimezone(timezone.utc)


def validate_configuration():
    for identifier in (CATALOG, SCHEMA):
        if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier):
            raise ValueError(
                "Catalog/schema must contain only letters, digits and underscores"
            )

    if not APPLICATION_CONTACT.strip():
        raise ValueError(
            "Set APPLICATION_CONTACT to a real company email or contact website"
        )

    if not isinstance(LOCATIONS, list) or not LOCATIONS:
        raise ValueError("Configure at least one location")

    locations = []
    location_ids = set()
    coordinates = set()

    for location in LOCATIONS:
        location_id = location.get("location_id")
        latitude = location.get("latitude")
        longitude = location.get("longitude")

        if (
            not isinstance(location_id, str)
            or not location_id.strip()
            or location_id in location_ids
        ):
            raise ValueError("Location IDs must be nonempty and unique")

        for value in (latitude, longitude):
            if (
                isinstance(value, bool)
                or not isinstance(value, (int, float))
                or not math.isfinite(value)
            ):
                raise ValueError(
                    f"{location_id}: coordinates must be finite numbers"
                )

        if not (-90 <= latitude <= 90 and -180 <= longitude <= 180):
            raise ValueError(f"{location_id}: invalid WGS84 coordinates")

        # MET requests must use at most four decimal places.
        latitude = round(latitude, 4)
        longitude = round(longitude, 4)

        if (latitude, longitude) in coordinates:
            raise ValueError("Use one location ID per coordinate pair")

        location_ids.add(location_id)
        coordinates.add((latitude, longitude))

        locations.append({
            "location_id": location_id,
            "latitude": latitude,
            "longitude": longitude,
        })

    return locations


def create_session():
    session = requests.Session()

    session.headers.update({
        "User-Agent": USER_AGENT,
        "Accept": "application/json",
    })

    retry_policy = Retry(
        total=4,
        backoff_factor=2,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
        respect_retry_after_header=True,
    )

    session.mount(
        "https://",
        HTTPAdapter(max_retries=retry_policy),
    )

    return session


def get_cached_record(request_key):
    timestamp_columns = {
        "source_updated_at",
        "first_ingested_at",
        "last_checked_at",
        "expires_at",
    }

    table = spark.table(RAW_TABLE)

    # Collect timestamps as epoch seconds to avoid driver timezone differences.
    columns = [
        F.col(name).cast("long").alias(name)
        if name in timestamp_columns
        else F.col(name)
        for name in table.columns
    ]

    rows = (
        table
        .where(F.col("request_key") == request_key)
        .orderBy(
            F.col("last_checked_at").desc(),
            F.col("record_id").desc(),
        )
        .select(*columns)
        .limit(1)
        .collect()
    )

    if not rows:
        return None

    record = rows[0].asDict()

    for name in timestamp_columns:
        value = record[name]
        record[name] = (
            datetime.fromtimestamp(value, timezone.utc)
            if value is not None
            else None
        )

    return record


def save_record(record):
    source = spark.createDataFrame([record], schema=RAW_SCHEMA)

    (
        DeltaTable.forName(spark, RAW_TABLE)
        .alias("target")
        .merge(
            source.alias("source"),
            "target.record_id = source.record_id",
        )
        .whenMatchedUpdate(set={
            "last_checked_at": "source.last_checked_at",
            "expires_at": "source.expires_at",
            "last_modified": "source.last_modified",
            "http_status": "source.http_status",
        })
        .whenNotMatchedInsertAll()
        .execute()
    )


def ingest_location(session, location):
    location_id = location["location_id"]

    params = {
        "lat": f"{location['latitude']:.4f}",
        "lon": f"{location['longitude']:.4f}",
    }

    source_url = requests.Request(
        "GET",
        API_URL,
        params=params,
    ).prepare().url

    request_key = hashlib.sha256(
        f"{location_id}|{source_url}".encode("utf-8")
    ).hexdigest()

    cached = get_cached_record(request_key)

    if cached and cached["expires_at"]:
        if utc_now() < cached["expires_at"]:
            print(
                f"{location_id}: cached until "
                f"{cached['expires_at'].isoformat()}; request skipped"
            )
            return "cached"

    headers = {}

    if cached and cached["last_modified"]:
        headers["If-Modified-Since"] = cached["last_modified"]

    time.sleep(REQUEST_INTERVAL_SECONDS)

    print(f"{location_id}: requesting forecast")

    with session.get(
        source_url,
        headers=headers,
        timeout=REQUEST_TIMEOUT,
    ) as response:
        response.raise_for_status()

        checked_at = utc_now()
        expires_at = parse_http_date(response.headers.get("Expires"))

        if expires_at is None:
            expires_at = checked_at + timedelta(
                minutes=CACHE_FALLBACK_MINUTES
            )
            print(
                f"{location_id}: Expires header missing; "
                f"using {CACHE_FALLBACK_MINUTES}-minute cache fallback"
            )

        if response.status_code == 304:
            if cached is None:
                raise ValueError("HTTP 304 received without a cached payload")

            record = dict(cached)
            record.update({
                "last_checked_at": checked_at,
                "expires_at": expires_at,
                "last_modified": (
                    response.headers.get("Last-Modified")
                    or cached["last_modified"]
                ),
                "http_status": 304,
            })

            save_record(record)

            print(f"{location_id}: forecast unchanged; cache metadata updated")
            return "unchanged"

        if response.status_code not in (200, 203):
            raise ValueError(
                f"Unexpected HTTP status: {response.status_code}"
            )

        if response.status_code == 203:
            print(
                "WARNING: MET returned HTTP 203; "
                "review product deprecation or beta status"
            )

        payload = response.json()
        properties = payload["properties"]
        timeseries = properties.get("timeseries")

        if not isinstance(timeseries, list) or not timeseries:
            raise ValueError("Response contains no forecast timeseries")

        updated_at = datetime.fromisoformat(
            properties["meta"]["updated_at"].replace("Z", "+00:00")
        )

        if updated_at.tzinfo is None:
            raise ValueError(
                "Forecast update timestamp must include a timezone"
            )

        canonical_payload = json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        )

        # Stable keys avoid duplicates when identical payloads are downloaded.
        record_id = hashlib.sha256(
            f"{request_key}|{canonical_payload}".encode("utf-8")
        ).hexdigest()

        record = {
            "record_id": record_id,
            "request_key": request_key,
            **location,
            "source_url": source_url,
            "source_updated_at": updated_at,
            "first_ingested_at": checked_at,
            "last_checked_at": checked_at,
            "expires_at": expires_at,
            "last_modified": response.headers.get("Last-Modified"),
            "http_status": response.status_code,
            "raw_payload": response.text,
        }

        save_record(record)

        print(
            f"{location_id}: stored {len(timeseries)} forecast points; "
            f"forecast updated {updated_at.isoformat()}"
        )

        return "received"


def initialize_storage():
    # The catalog must already exist.
    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`"
    )

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {RAW_TABLE}
        ({RAW_SCHEMA})
        USING DELTA
    """)

    # Preserve forecast versions and the source units.
    # Missing forecast intervals remain null.
    spark.sql(f"""
        CREATE OR REPLACE VIEW {FORECAST_VIEW} AS
        WITH parsed AS (
            SELECT
                *,
                from_json(
                    raw_payload,
                    '{FORECAST_JSON_SCHEMA}'
                ) AS forecast
            FROM {RAW_TABLE}
        ),
        expanded AS (
            SELECT
                record_id,
                location_id,
                latitude,
                longitude,
                source_updated_at,
                first_ingested_at,
                forecast.properties.meta.units AS units,
                explode(forecast.properties.timeseries) AS point
            FROM parsed
        )
        SELECT
            record_id,
            location_id,
            latitude,
            longitude,
            source_updated_at,
            first_ingested_at,
            to_timestamp(point.time) AS forecast_time_utc,
            units,
            point.data['instant'].details AS instant_details,
            point.data['next_1_hours'].details AS next_1_hour_details,
            point.data['next_6_hours'].details AS next_6_hours_details,
            point.data['next_12_hours'].details AS next_12_hours_details,
            point.data['next_1_hours'].summary['symbol_code']
                AS next_1_hour_symbol
        FROM expanded
    """)

### Main code

In [0]:
locations = validate_configuration()

spark.conf.set("spark.sql.session.timeZone", "UTC")
initialize_storage()

print(f"Starting MET Norway ingestion for {len(locations)} location(s)")
print(f"Raw Delta table: {RAW_TABLE}")
print(f"Forecast view: {FORECAST_VIEW}")

counts = {
    "cached": 0,
    "unchanged": 0,
    "received": 0,
}
failures = []

with create_session() as session:
    for location in locations:
        try:
            result = ingest_location(session, location)
            counts[result] += 1

        except Exception as exc:
            failures.append(location["location_id"])
            print(
                f"ERROR: {location['location_id']}: "
                f"{type(exc).__name__}: {exc}"
            )

print(f"Ingestion results: {counts}")
print(f"Failed locations: {len(failures)}")

if failures:
    raise RuntimeError(
        f"Ingestion failed for: {failures}. "
        "Successful writes are retained; rerun to retry."
    )

print("MET Norway ingestion completed successfully")